# Chapter 20 — Data cleaning, profiling, validation, and quality reporting

**Book role:** Analytics foundations  
**Suggested book space:** about 10 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Profile missing values, duplicates, types, and category inconsistencies.
- Design a repeatable cleaning pipeline with an audit trail.
- Separate warnings from rejection rules.
- Present data-quality findings to users before destructive transformations.

## 1. Profile before modifying

Start with a quality profile: shape, column names, dtypes, missing-value counts and rates, duplicate rows, duplicate business keys, distinct-category counts, and numerical ranges. Unexpected values may be legitimate (a negative amount for a return) or indicate an error (an impossible quantity). Compare observations with domain rules rather than applying generic “cleaning” automatically.

A profiling report should capture both the issue and its scope. “Quantity has missing values in 2.3% of rows” is more useful than “some data is missing.” Show example rows for review, but avoid dumping sensitive values into log files or reports. Keep the raw DataFrame unchanged until a cleaning decision is confirmed or persisted as a repeatable transformation.

### Worked example 1: Build a compact profile report

Exact row duplicates and duplicate business keys are different checks. A repeated order ID may represent multiple legitimate line items, so the uniqueness constraint must match the data grain.

In [ ]:
import pandas as pd

frame = pd.DataFrame({
    "order_id": [1, 2, 2, 4],
    "region": ["North", " north ", None, "West"],
    "quantity": [2, None, 1, -3],
    "revenue": [200.0, 0.0, 150.0, -300.0],
})
profile = pd.DataFrame({
    "dtype": frame.dtypes.astype(str),
    "missing_count": frame.isna().sum(),
    "missing_pct": (frame.isna().mean() * 100).round(2),
    "distinct_non_null": frame.nunique(dropna=True),
})
print("Shape:", frame.shape)
print("Exact duplicate rows:", int(frame.duplicated().sum()))
print(profile)
print("Potential duplicate order IDs:", int(frame["order_id"].duplicated(keep=False).sum()))

## 2. Cleaning decisions should be auditable

Common actions include trimming text, standardizing category labels, parsing dates, coercing numeric values under a controlled policy, removing true duplicates, and deriving measures. Each transformation should have a reason and a test. Do not replace all missing numeric values with zero unless zero is the actual business meaning; a missing cost is not necessarily cost-free.

Track how many rows and values each step changes. Prefer a pipeline that returns a cleaned DataFrame plus an issue report or metadata object. That makes it possible to tell a user how many rows were changed or rejected, to compare before/after summaries, and to reproduce results. Where appropriate, keep a rejected-rows file with reason codes and a secure retention policy.

### Worked example 2: Make cleaning steps measurable

The coercion policy converts malformed numeric text to missing values so it can be flagged; it does not silently make those records zero. In production, count failures against the original non-null mask and attach row identifiers and rule names.

In [ ]:
import pandas as pd

raw = pd.DataFrame({"region": [" North ", "north", "WEST", None], "quantity": ["2", "bad", "1", "3"]})
clean = raw.copy()
clean["region"] = clean["region"].astype("string").str.strip().str.title()
clean["region"] = clean["region"].replace({"<NA>": pd.NA, "North": "North", "West": "West"})
raw_quantity_missing = raw["quantity"].isna().sum()
clean["quantity"] = pd.to_numeric(clean["quantity"], errors="coerce")
new_coercion_failures = int(clean["quantity"].isna().sum() - raw_quantity_missing)
print(clean)
print("New numeric coercion failures:", new_coercion_failures)

## 3. Validation schema and rule severity

Define required columns, expected dtypes, allowed categorical values, non-null constraints, key uniqueness, valid ranges, and cross-column checks. A warning may allow import with a visible caution; an error may require a correction or an explicit override. Examples include unknown categories (warning), missing transaction IDs (error), and profit not matching revenue minus cost beyond a rounding tolerance (error).

Quality is continuous: a later file may contain new categories or a format shift. Avoid embedding all rules inside a button callback. Put them in a testable validation module, version the rule set, and display the version in the import summary if results need to be auditable.

## Guided lab: Lab: quality report tab

Build a report containing row count, missingness, duplicate keys, invalid dates, negative quantity count, unexpected categories, and revenue/profit consistency. For each issue, show severity, affected row count, an example, and the proposed remedy. Add “Continue with warnings” and “Cancel import” behavior, but do not let the GUI override hard validation errors without a documented rule.

## Review questions and practice

1. Why are duplicate rows and duplicate keys different?
2. Why is replacing missing values with zero dangerous?
3. Which issues should block an import versus warn the user?
4. Why should every cleaning transformation be measured?
5. What should an issue report include besides a description?

## Further reading

- pandas missing data: https://pandas.pydata.org/docs/user_guide/missing_data.html
- pandas duplicate detection: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.